# Reproducing Zhu et al. (2022) from dandiset 001193

**Paper.** Zhu Y, Xu M, Lu J, Hu J, Kwok VPY, Zhou Y, Yuan D, Wu B, Zhang J, Wu J, Tan LH (2022).
Distinct spatiotemporal patterns of syntactic and semantic processing in human inferior frontal
gyrus. *Nature Human Behaviour* 6, 1104-1111.
[doi:10.1038/s41562-022-01334-6](https://doi.org/10.1038/s41562-022-01334-6).

**Claims file.** `claims/001193/10.1038-s41562-022-01334-6.yaml`, written from the version of
record (main text, figure legends, Methods, reporting summary) and the Supplementary Information
PDF, and checked by a second reader. The paper links no code.

**Dandiset.** [DANDI:001193](https://dandiarchive.org/dandiset/001193/0.240921.1006), version
0.240921.1006 (doi:10.48324/dandi.001193/0.240921.1006). The paper cites no version of the
dandiset (its data statement says the data are available on request; the dandiset was made two
years after publication). This is the only published version, so it is pinned. The draft holds
the same 40 files with the same checksums and sizes, and its metadata differ only in the fields
that name the version (checked below), so the choice does not change any result.

**What the dandiset holds.** One NWB file per experimental block: 10 participants, 4 blocks each.
Each file holds a single 256-channel high-gamma series and an electrode table. It holds no trial
or event times, no condition labels, no button responses, no electrode locations (anatomical
labels or MNI coordinates), no broadband signal or other frequency band, and no per-participant
sex or age. Every claim on the recordings except the participant counts needs at least one of
these, so 23 claims are blocked as `data_missing`.

**Outside inputs.** One claim, stimuli-word-frequency (`data: with_inputs`), uses no data of the
dandiset: it is tested on the sentence lists of the paper's Supplementary Information, fetched from
the publisher and checked against the copy the claims file was written from (SHA-256), and on the
bigram counts of the corpus the Methods name (Jun Da, Chinese text computing,
lingua.mtsu.edu/chinese-computing), read from the site's bigram search, with the sub-corpus sizes
of the technical report the site cites (Da 2004). The site has no version, DOI or commit: its
counts are as served on the day the notebook runs, and they are printed below.

| Claim | Tested here | Why not |
|---|---|---|
| participants | yes | |
| task-structure | no | no trial or word event times, no condition labels |
| behaviour-response-time | no | no button responses or question onsets |
| fig1b-high-gamma-best-band | no | only high gamma; no other band, no broadband; no events |
| fig1c-responsive-in-ifg | no | no events (rest against stimulus), no electrode locations |
| fig1d-only-after-keyword | no | no events or condition labels |
| fig2b-violation-electrodes | no | no events or condition labels, no electrode locations |
| fig2b-single-type | no | depends on fig2b-violation-electrodes |
| suppfig6a-half-split | no | no events, labels or sentence identities |
| suppfig6b-duration-metric | no | no events or condition labels |
| fig2c-euclidean-similarity | no | depends on fig2b-violation-electrodes |
| suppfig7a-com-overlap | no | no COM labels, no events |
| suppfig7b-not-nonlinguistic | no | no NL labels, no events |
| fig2de-no-spatial-separation | no | no MNI coordinates; depends on fig2b-violation-electrodes |
| suppfig8b-individual-no-separation | no | no coordinates; depends on fig2b-violation-electrodes |
| fig2f-circuit-diameter | no | depends on fig2b-single-type |
| suppfig9-predict-from-corr | no | no CORR labels or events; depends on fig2b-single-type |
| fig3-latency-values | no | no keyword onsets; depends on fig2b-violation-electrodes |
| fig3d-latency-differences | no | depends on fig3-latency-values |
| fig3-sync-latency-variability | no | depends on fig3-latency-values |
| fig3e-latency-not-location | no | no coordinates; depends on fig3-latency-values |
| suppfig10a-separable-400-700 | no | no condition labels or keyword onsets |
| suppfig10b-duration-differs | no | depends on fig2b-violation-electrodes |
| nl-low-frequency | no | no 4 to 10 Hz signal, no NL labels |
| stimuli-word-frequency | yes, outside inputs only | |

In [ ]:
import time
import json

import numpy as np
import pandas as pd

import ifg_hga

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

started = time.time()
versions = ifg_hga.published_versions()
meta, files = ifg_hga.version_metadata()
draft_meta, draft_files = ifg_hga.version_metadata("draft")
summaries = ifg_hga.load_all()
print(f"dandiset {ifg_hga.DANDISET_ID}, version {ifg_hga.DANDISET_VERSION}: read {len(summaries)} files "
      f"in {time.time() - started:.0f} s")

## Data check

Each statement about the metadata is checked against the values under the output it rests on.

In [ ]:
print("Versions the archive lists:", versions)
print("Files at the pinned version:", len(files), "; in the draft:", len(draft_files))
print("Draft files identical to the pinned version (path, checksum, size):", files == draft_files)
skip = {"id", "version", "doi", "url", "citation", "manifestLocation", "dateModified", "datePublished",
        "publishedBy", "assetsSummary"}
differ = sorted(k for k in set(meta) | set(draft_meta) if k not in skip and meta.get(k) != draft_meta.get(k))
print("Other metadata fields that differ between the pinned version and the draft:", differ)
print("\nDescription:", meta["description"])
summary = meta["assetsSummary"]
print("\nArchive summary: subjects", summary.get("numberOfSubjects"), "; files", summary.get("numberOfFiles"),
      "; bytes", summary.get("numberOfBytes"), "; variables", summary.get("variableMeasured"))
print("Related resources in the metadata:", meta.get("relatedResource"))

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ("path", "identifier", "subject_id", "session_start", "rate",
                                         "n_samples", "n_channels", "unit", "sex", "age")} for s in summaries])
info["block"] = info.identifier.str.split("_block").str[1].astype(int)
info["session_date"] = info.session_start.str[:10]
print("Files per subject and block (identifier field):")
print(pd.crosstab(info.subject_id, info.block).to_string())
print("\nDistinct subjects:", info.subject_id.nunique(), "; files:", len(info))
print("Each file's identifier names its subject:", bool((info.identifier.str.split("_block").str[0] == info.subject_id).all()))
print("One session date per subject:", bool(info.groupby("subject_id").session_date.nunique().eq(1).all()))
print("\nPer-subject metadata (identical in every file?)")
for key in ("sex", "age", "subject_description", "species", "session_description", "experiment_description",
            "related_publications", "series_description", "unit", "conversion", "rate", "starting_time"):
    values = sorted({json.dumps(s[key], ensure_ascii=False) for s in summaries})
    print(f"  {key}: {', '.join(values)}")
print("\nSubjects by session date:")
print(info.groupby("subject_id").session_date.first().sort_values().to_string())

The files hold 10 subjects with four blocks each, as the Methods' four blocks per participant.
Every file gives sex `U` and age `P19Y/P50Y` (the age range of the whole group), with the
subject description "chinese, 4 males and 6 females" in every file. Supplementary Table 1 gives
each participant's sex and age, but nothing in the dandiset maps the subject ids (TC5 to TC20)
to the participants numbered 1 to 10 there, so per-participant sex and age cannot be read or
checked.

In [ ]:
print("Objects outside acquisition and general, per file:", sorted({json.dumps(s["other_objects"]) for s in summaries}))
print("Files with a trials table:", sum(s["has_trials"] for s in summaries))
print("Electrode table columns:", sorted({json.dumps(s["electrode_columns"]) for s in summaries}))
print("Values of `location`:", sorted({v for s in summaries for v in s["locations"]}))
print("Values of `filtering`:", sorted({v for s in summaries for v in s["filtering"]}))
print("Electrodes per group:", sorted({json.dumps({str(k): v for k, v in s["group_counts"].items()}) for s in summaries}))
print("Group descriptions:", sorted({json.dumps(s["group_descriptions"]) for s in summaries}))
print("Group locations:", sorted({json.dumps(s["group_locations"]) for s in summaries}))
print("Series rows 0..255 in order in every file:", all(s["series_electrodes"] == list(range(256)) for s in summaries))
listings = {}
for s in summaries:
    key = (tuple(s["top_level"]),
           tuple(l.replace(f"({s['n_samples']}, 256)", "(n_samples, 256)") for l in s["acquisition_listing"] + s["general_listing"]))
    listings.setdefault(key, []).append(s["identifier"])
print("\nDistinct listings of the top level, `acquisition` and `general` (every object, with its neurodata type):", len(listings))
for (top, listing), ids in listings.items():
    print(f"  shared by {len(ids)} files ({len({i.split('_')[0] for i in ids})} subjects)")
    print("  top level:", list(top))
    for line in listing:
        print("   ", line)
print("Storage of the ECoG data:", sorted({json.dumps(s["data_storage"]) for s in summaries}))


In [ ]:
sizes = pd.DataFrame([{"path": s["path"], "n_samples": s["n_samples"], "asset_size": s["asset_size"]} for s in summaries])
sizes["data_bytes"] = sizes.n_samples * 256 * 4
sizes["other_bytes"] = sizes.asset_size - sizes.data_bytes
print("File size against the ECoG data (n_samples x 256 channels x 4 bytes, float32, uncompressed):")
print(sizes.to_string(index=False))
print("\nBytes other than the ECoG data, over the 40 files:", sorted(sizes.other_bytes.unique().tolist()))


Apart from the high-gamma series, every file holds only empty `stimulus`, `analysis` and
`processing` groups: no trials table, no intervals, no events, no photodiode, no responses. The
full listing of `acquisition` and `general` above (one listing for all 40 files) holds only the
`ECoG` ElectricalSeries, the devices, the two electrode groups, the electrode table and the
subject and session text: no TimeSeries of events, annotations or other series. The table of
sizes above agrees: every file holds the same number of bytes beyond its uncompressed ECoG data
(printed above), so no file carries an event table of its own. The electrode table has no
coordinates, every `location` is `unknown`, and neither grid says which cortex it covers
(Supplementary Table 1: one grid over the IFG, the other over the MFG or STG). The analyses of
the paper start from keyword onsets, condition labels and IFG labels, none of which is here.


In [ ]:
vals = pd.DataFrame([{k: s[k] for k in ("identifier", "n_nan", "channel_mean_absmax", "channel_std_min",
                                         "channel_std_max", "min", "p01", "p50", "p99", "max", "frac_above_10",
                                         "first_sample_median", "start_transient_samples", "n_duplicate_channels")} for s in summaries])
print("Values per file (z units if the data are z scores):")
print(vals.round(4).to_string(index=False))
print("\nLargest |channel mean| over all files:", f"{vals.channel_mean_absmax.max():.2e}")
print("Channel standard deviations over all files:", round(vals.channel_std_min.min(), 5), "to", round(vals.channel_std_max.max(), 5))
print("NaN values:", int(vals.n_nan.sum()), "; identical channel pairs within a file:", int(vals.n_duplicate_channels.sum()))
starts = vals[vals.first_sample_median >= 3]
print("Files whose first sample has a channel median of 3 z or more:", len(starts), sorted(set(starts.identifier.str.split("_").str[0])))
print("  samples until the channel median falls below 3 z:", int(starts.start_transient_samples.min()), "to",
      int(starts.start_transient_samples.max()))

The values are stated in volts, but every channel of every file has mean 0 (largest absolute
mean printed above) and standard deviation 1 to within 3e-5. They are z scores per channel and
per file, consistent with the Methods' HGA "z scored relative to the mean and standard deviation
of the data within each experimental block", and with each file being one block. The unit
`volts` is wrong. Because each file is z scored on its own, no channel can be identified as a bad
channel from its scale, and the dandiset does not say which channels or intervals were excluded
(the Methods exclude both); all 256 channels are present in every file. Some files (counted
above; all those of TC5, TC6, TC7 and TC9) begin with a transient shared by all channels, with a
channel median of up to 50 z at the first sample, which falls below 3 z within the few samples
printed above.

In [ ]:
rate_check = info.groupby("subject_id").agg(blocks=("block", "size"), samples_min=("n_samples", "min"),
                                             samples_max=("n_samples", "max"))
for hz in (400.0, ifg_hga.PAPER_HGA_HZ):
    rate_check[f"block_s_at_{int(hz)}Hz_min"] = rate_check.samples_min / hz
    rate_check[f"block_s_at_{int(hz)}Hz_max"] = rate_check.samples_max / hz
    rate_check[f"s_per_trial_at_{int(hz)}Hz_max"] = rate_check.samples_max / hz / ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK
need = ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK * ifg_hga.MIN_TRIAL_S
print(f"A block holds at least {ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK} linguistic trials of at least "
      f"{ifg_hga.MIN_TRIAL_S:.1f} s before the question: at least {need:.0f} s.")
print(rate_check.round(1).to_string())
print(f"\nBlocks shorter than {need:.0f} s at the stated 400 Hz:", int((info.n_samples / 400 < need).sum()), "of", len(info))
print(f"Blocks shorter than {need:.0f} s at the paper's {ifg_hga.PAPER_HGA_HZ:.0f} Hz:",
      int((info.n_samples / ifg_hga.PAPER_HGA_HZ < need).sum()), "of", len(info))

excess = pd.DataFrame([{"identifier": s["identifier"], "word_rate_if_400Hz": s["excess_word_rate_if_400"],
                        "word_rate_if_100Hz": s["excess_word_rate_if_100"]} for s in summaries])
print("\nPower at the 2 Hz word rate over the flanking band (log10, mean over channels), under each rate:")
print(excess.describe().loc[["mean", "min", "50%", "max"]].round(3).to_string())
print("Files with excess above 0.1 (a 26% peak):", {c: int((excess[c] > 0.1).sum()) for c in excess.columns[1:]})
print("Files with any excess above 0:", {c: int((excess[c] > 0).sum()) for c in excess.columns[1:]}, "of", len(excess))

The series are stated at 400 Hz. At that rate the blocks last 70 to 95 s, under 3 s per
linguistic trial, while the task needs at least 4.3 s per trial before the question; every block
is too short (count above). At 100 Hz, the rate the Methods give for the HGA, the blocks last 282
to 381 s, 8.8 to 11.9 s per trial, which the task allows. The data are consistent with a rate of
100 Hz, not 400 Hz, unless each file holds less than half of its block (the Methods exclude bad
intervals, but say nothing of that size). The 400 Hz is the rate the Methods give for the signal
before the HGA was taken. The spectral check at the 2 Hz word rate leans the same way but is
weak: at the period the 100 Hz reading predicts, the power is above its flanking band in 34 of 40
files (median excess 0.021 log10, about 5%), against 19 of 40 (median -0.001) at the period the
400 Hz reading predicts; only one file shows a peak above 0.1 under either reading. The blocks'
length is the stronger argument. Neither check shows the rate directly. Without event times, no stream can be shown to share a
clock with a trial table: this could not be established for any session. An analysis of these
files would take them at 100 Hz, the rate of the Methods' HGA, the rate the block lengths support;
no claim here uses time, so the choice changes no result.

In [ ]:
names = [s["identifier"] for s in summaries]
match = np.array([[ifg_hga.best_match(a["fingerprint"], b["trace"]) if a is not b else np.nan for b in summaries]
                  for a in summaries])
pairs = [(names[i], names[j], round(match[i, j], 4)) for i in range(len(names)) for j in range(len(names))
         if i != j and match[i, j] > 0.9]
print(f"Largest correlation of {ifg_hga.FINGERPRINT_SAMPLES} samples from the middle of one file (differenced "
      "channel mean) with any stretch of another file:")
print("  over all pairs, max", round(np.nanmax(match), 4), "; median", round(np.nanmedian(match), 4))
print("  pairs above 0.9:", pairs)
i, j = np.unravel_index(np.nanargmax(match), match.shape)
print("  most similar pair:", names[i], names[j])
print("Distinct sample counts:", info.n_samples.nunique(), "of", len(info))
print("Self-match of each file (should be 1):", round(min(ifg_hga.best_match(s["fingerprint"], s["trace"]) for s in summaries), 6))

No block was deposited twice: no stretch from the middle of a file matches any stretch of
another file (largest correlation printed above, against 1 for a file with itself), and all 40
sample counts differ. Each recording is counted once.

## participants

> Ten participants (four male, six female, 19 to 50 years) were recorded with two 128-electrode
> grids each; all ten did the CORR, SYN-P, SYN-C and SEM conditions, five of them the combined
> violation (COM) and the other five the non-linguistic task (NL).

Criterion: counts of distinct subjects, their sex, and the tasks each has; tolerance zero;
reproduced when all match, partially reproduced when the dataset holds a subset with matching
metadata; ages compared with Supplementary Table 1 where the dataset gives them.

In [ ]:
per_subject = pd.DataFrame([{"subject_id": s["subject_id"], "sex": s["sex"], "age": s["age"],
                             "electrodes": s["n_channels"], "grid_1": s["group_counts"].get("Grid_1"),
                             "grid_2": s["group_counts"].get("Grid_2")} for s in summaries]).drop_duplicates()
print(per_subject.sort_values("subject_id").to_string(index=False))
n_subjects = per_subject.subject_id.nunique()
sex_values = sorted(per_subject.sex.unique())
known = per_subject.sex.isin(["M", "F"])
if known.all():
    males, females = int((per_subject.sex == "M").sum()), int((per_subject.sex == "F").sum())
else:
    males = females = (f"not determinable: sex is {sex_values} for {int((~known).sum())} of {len(per_subject)} "
                       "subjects")
print("\nExpected against found:")
rows = [
    ("participants", 10, n_subjects),
    ("male participants (sex field)", 4, males),
    ("female participants (sex field)", 6, females),
    ("participants with COM", 5, "no task or condition labels in the files"),
    ("participants with NL", 5, "no task or condition labels in the files"),
    ("electrodes per participant", 256, sorted(per_subject.electrodes.unique().tolist())),
]
for what, expected, found in rows:
    print(f"  {what}: expected {expected}, found {found}")
print("  sex as free text in every file's subject description:", sorted({s["subject_description"] for s in summaries}))
print("  grids per participant: 2 of 128 in every file:",
      bool(((per_subject.grid_1 == 128) & (per_subject.grid_2 == 128)).all()))
print("  ages: every file gives", sorted(per_subject.age.unique()), "; Supplementary Table 1 range is 19 to 50 years")

**Result: partially reproduced.** The dataset holds 10 distinct participants, each with 256
electrodes in two groups of 128, as the paper states. The sex field is `U` (unknown) for all 10, so
the numbers of male and female participants cannot be determined from the subjects; the 4 and 6
appear only as a sentence copied into every file's subject description. Ages are given only as
the group's range (19 to 50 years), which matches the paper's range but not per participant. No
file says which task a participant did, so the 5 COM and 5 NL participants cannot be counted (the
two recording periods, TC5 to TC12 and TC16 to TC20, are not evidence of the task groups). Two of
the six counts match from the data; the others are not in it.


## stimuli-word-frequency

> Keyword frequencies are matched between CORR (shared by SYN-P and SEM; mean 31.88 per million)
> and SYN-C (30.25 per million).

Criterion: keywords from the supplement's sentence lists (word 4 of CORR and SYN-C), frequencies
from the corpus the paper names (lingua.mtsu.edu/chinese-computing, the Modern Chinese word
frequency list). Reproduced when both means are within tolerance (31.88 ± 3.188 and
30.25 ± 3.025 per million) and an independent two-sample t-test gives p > 0.05.

This claim uses no data of the dandiset (`data: with_inputs`). The site has no word list as such:
its Modern Chinese list counts single characters, and its word-level counts are bigram
(two-character string) counts for two sub-corpora, news (14 million characters) and general
fiction (18 million). Every keyword has two characters (Methods, "All the keywords were of two
characters"), so the bigram counts are the site's frequencies of the keywords. The paper does not
say which sub-corpus it used, or whether "per million" is per million bigrams or characters, so
every combination is run. The t-test is Student's, as t(62) with 32 and 32 keywords implies; a
keyword that occurs twice in a list is counted twice for the same reason.


In [ ]:
import keyword_frequency as kf
from scipy import stats

raw, digest = kf.supplement()
print("Supplementary Information:", kf.SUPPLEMENT_URL)
print("  SHA-256", digest, "; the copy the claims file was written from:", digest == kf.SUPPLEMENT_SHA256)
lists = kf.sentence_lists(raw)
print("Sentences per list:", {k: len(v) for k, v in lists.items()})
print("Words per sentence:", {k: sorted({len(w) for w in v.values()}) for k, v in lists.items()})
numbers = range(1, 33)
keywords = pd.DataFrame({"CORR": [lists["CORR"][i][3] for i in numbers], "SYN-C": [lists["SYN-C"][i][3] for i in numbers],
                         "SYN-P": [lists["SYN-P"][i][4] for i in numbers], "SEM": [lists["SEM"][i][3] for i in numbers]},
                        index=pd.Index(numbers, name="sentence"))
print("Keywords of two characters:", {k: int(keywords[k].str.len().eq(2).sum()) for k in keywords}, "of 32")
for other in ("SYN-P", "SEM"):
    differ = keywords[keywords.CORR != keywords[other]]
    print(f"CORR keyword (word 4) equal to the {other} keyword in {32 - len(differ)} of 32;",
          "differing:", list(zip(differ.index, differ.CORR, differ[other])))
for k in ("CORR", "SYN-C"):
    repeated = keywords[k][keywords[k].duplicated(keep=False)]
    print(f"Keywords occurring twice in {k}:", sorted(set(repeated)), "at sentences", list(repeated.index))

corpus, table = kf.corpus_sizes()
print("\nSub-corpus sizes, Table 6 of Da (2004),", kf.REPORT_URL)
print(pd.DataFrame(corpus).T.to_string())

counts = kf.all_counts(pd.unique(keywords.values.ravel()))
absent = {w: c["absent_from"] for w, c in counts.items() if c["absent_from"]}
print("\nKeywords absent from a sub-corpus (the site answers '(No info.)'; counted as 0):", absent)
raw_counts = pd.DataFrame({k: {"news": [counts[w]["news"] for w in keywords[k]],
                               "fiction": [counts[w]["fiction"] for w in keywords[k]]} for k in ("CORR", "SYN-C")})
shown = pd.DataFrame({"CORR": keywords.CORR, "CORR news": [counts[w]["news"] for w in keywords.CORR],
                      "CORR fiction": [counts[w]["fiction"] for w in keywords.CORR],
                      "SYN-C": keywords["SYN-C"], "SYN-C news": [counts[w]["news"] for w in keywords["SYN-C"]],
                      "SYN-C fiction": [counts[w]["fiction"] for w in keywords["SYN-C"]]}, index=keywords.index)
print("\nRaw bigram counts as served by", kf.SEARCH_URL, "on", time.strftime("%Y-%m-%d", time.gmtime()))
print(shown.to_string())


In [ ]:
EXPECTED = {"CORR": (31.88, 3.188), "SYN-C": (30.25, 3.025)}   # the claim's expected values and tolerances
PAPER_SD = {"CORR": 35.14, "SYN-C": 73.58}                     # the Methods' s.d., for comparison only
READINGS = {"news": ("news",), "fiction": ("fiction",), "news + fiction": ("news", "fiction")}


def per_million(words, subs, total):
    n = sum(corpus[s][total] for s in subs)
    return np.array([1e6 * sum(counts[w][s] for s in subs) / n for w in words])


def test(corr_words, sync_words, subs, total, equal_var=True):
    a, b = per_million(corr_words, subs, total), per_million(sync_words, subs, total)
    t, p = stats.ttest_ind(a, b, equal_var=equal_var)
    pooled = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return {"CORR mean": a.mean(), "CORR sd": a.std(ddof=1), "SYN-C mean": b.mean(), "SYN-C sd": b.std(ddof=1),
            "t": t, "df": len(a) + len(b) - 2, "p": p, "d": (a.mean() - b.mean()) / pooled,
            "CORR in tol": abs(a.mean() - EXPECTED["CORR"][0]) <= EXPECTED["CORR"][1],
            "SYN-C in tol": abs(b.mean() - EXPECTED["SYN-C"][0]) <= EXPECTED["SYN-C"][1],
            "p > 0.05": p > 0.05}


rows = []
for name, subs in READINGS.items():
    for total in ("bigrams", "characters"):
        rows.append({"sub-corpus": name, "per million": total, **test(keywords.CORR, keywords["SYN-C"], subs, total)})
readings = pd.DataFrame(rows)
readings["criterion met"] = readings[["CORR in tol", "SYN-C in tol", "p > 0.05"]].all(axis=1)
print("Every reading of the corpus (keywords: word 4 of CORR and SYN-C; Student's t-test):")
print(readings.round(3).to_string(index=False))
print("Paper: CORR 31.88 (s.d. 35.14), SYN-C 30.25 (s.d. 73.58), t(62) = 0.11, P = 0.91;",
      "tolerance bands", {k: (round(v - tol, 3), round(v + tol, 3)) for k, (v, tol) in EXPECTED.items()})
print("Readings meeting the criterion:", readings.loc[readings["criterion met"], ["sub-corpus", "per million"]].values.tolist(),
      "of", len(readings))

main = readings[(readings["sub-corpus"] == "news + fiction") & (readings["per million"] == "bigrams")].iloc[0]
ratios = {"CORR mean": 31.88 / main["CORR mean"], "SYN-C mean": 30.25 / main["SYN-C mean"],
          "CORR sd": PAPER_SD["CORR"] / main["CORR sd"], "SYN-C sd": PAPER_SD["SYN-C"] / main["SYN-C sd"]}
print("\nPooled reading (news + fiction, per million bigrams), paper value / value here:",
      {k: round(float(v), 4) for k, v in ratios.items()})
pooled_bigrams = corpus["news"]["bigrams"] + corpus["fiction"]["bigrams"]
print(f"Bigrams in the two sub-corpora: {pooled_bigrams:,}; the total the paper's values imply "
      f"(this total / mean ratio): {pooled_bigrams / np.mean(list(ratios.values())):,.0f}")


In [ ]:
variants = []
dedup = lambda words: list(dict.fromkeys(words))
for name, subs in READINGS.items():
    for label, corr_words, sync_words, eq in (
            ("SYN-P keywords (word 5) for CORR", keywords["SYN-P"], keywords["SYN-C"], True),
            ("SEM keywords (word 4) for CORR", keywords["SEM"], keywords["SYN-C"], True),
            ("each keyword once per list", dedup(keywords.CORR), dedup(keywords["SYN-C"]), True),
            ("keywords absent from the news sub-corpus dropped",
             [w for w in keywords.CORR if w not in absent], [w for w in keywords["SYN-C"] if w not in absent], True),
            ("Welch's t-test", keywords.CORR, keywords["SYN-C"], False)):
        r = test(corr_words, sync_words, subs, "bigrams", equal_var=eq)
        variants.append({"sub-corpus": name, "variant": label, **r})
variants = pd.DataFrame(variants)
variants["criterion met"] = variants[["CORR in tol", "SYN-C in tol", "p > 0.05"]].all(axis=1)
print("Variants, per million bigrams (df is n1 + n2 - 2 even for Welch's test):")
print(variants.round(3).to_string(index=False))


**Result: reproduced, on an assumed reading of the corpus.** With the two sub-corpora of the
site pooled and counts per million bigrams, the CORR keywords have a mean of 31.57 per million
(s.d. 34.80) and the SYN-C keywords 29.95 (s.d. 72.86), both within tolerance of the paper's 31.88
and 30.25, and Student's t-test gives t(62) = 0.11, p = 0.910, as the paper's t(62) = 0.11,
P = 0.91. The paper's two means and two s.d. are each 1.010 times the values here, as if the
paper divided the same counts by about 28.0 million where the two sub-corpora hold 28.3 million
bigrams. No other reading meets the criterion: the news sub-corpus alone puts the CORR mean below
its band, the fiction sub-corpus alone puts the SYN-C mean below its band (and the CORR mean above
it per million bigrams), and the pooled counts per million characters put both means below.
The pooled reading is the assumption: the paper names the site, not a sub-corpus, the site's
search returns both sub-corpora together, and bigram counts are counts of bigrams. Under every
reading and variant printed the means do not differ (p from 0.139 to 0.964). The paper's values
are those of the CORR list's keywords: taking the SYN-P (or SEM) keywords, which differ from
CORR's in sentences 3 and 11, raises the pooled CORR mean to 47.35, outside the band, although the
Methods say the keywords are the same. The site's counts are not versioned; they are those served
on the day printed above.


## Summary

| Claim | Outcome |
|---|---|
| participants | partially reproduced |
| the other 23 claims on the recordings | blocked, data missing |
| stimuli-word-frequency (outside inputs only) | reproduced, on the assumed pooled reading of the corpus |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Of the recordings, only the counts of participants
and electrodes are tested. Every analysis of the paper aligns the high-gamma activity to keyword
onsets and compares conditions within IFG electrodes; the dataset holds none of the event times,
condition labels or electrode locations these need. The keyword frequencies use the site's bigram
counts of the two sub-corpora pooled, per million bigrams, an assumption, as the paper does not
say which list or which total it used; the site's counts are not versioned.

**What was found about the dataset.**

- No trial, word or keyword event times, condition labels, button responses or photodiode signal.
- No electrode locations: every `location` is `unknown`, no coordinates, and the two grids do not
  say which cortex each covers.
- Only high gamma, so the frequency-band analyses (Fig. 1b, Supplementary Figs. 3, 4, 7c) cannot
  be repeated.
- Sex `U` and age `P19Y/P50Y` for every subject, and no mapping from the subject ids to the
  participants of Supplementary Table 1.
- The unit `volts` is wrong: the values are z scores per channel and block.
- The stated rate of 400 Hz is inconsistent with the length of the blocks and consistent with
  the paper's 100 Hz HGA.
- No record of the bad channels and intervals the Methods exclude.
- No recording deposited twice.